In [ ]:
!pip install -q transformers datasets sentencepiece accelerate evaluate optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 38.9 MB/s eta 0:00:00


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import optuna
import gc
from pathlib import Path
from sklearn.model_selection import train_test_split,KFold
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5Tokenizer,
    T5ForConditionalGeneration,
    set_seed
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Device: cuda


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CSV_PATH = "/content/drive/MyDrive/ASQP/saida_asqp/treino_tratado.csv"

df = pd.read_csv(CSV_PATH)

TEXT_COL = "text"
TARGET_COL = "target"

if "input" not in df.columns:
    df["input"] = "extrair quadruplas: " + df[TEXT_COL].astype(str)

df = df.dropna(subset=["input", TARGET_COL]).copy()

df["input"] = df["input"].astype(str)
df[TARGET_COL] = df[TARGET_COL].astype(str)

print("Total:", len(df))
display(df[["input", TARGET_COL]].head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total: 730


,input,target
0,extrair quadruplas:hotel com condições gerais ...,[A] hotel [C] general [O] más [P] neg [SSEP] [...
1,extrair quadruplas:o hotel segue corretamente ...,[A] localização [C] location [O] excelente [P]...
2,extrair quadruplas:o hotel é muito luxuoso e e...,[A] hotel [C] general [O] luxuoso [P] pos [SSE...
3,extrair quadruplas:o hotel em geral é bom o qu...,[A] hotel [C] general [O] bom [P] pos [SSEP] [...
4,extrair quadruplas:em cidades grandes e com mu...,[A] hotel [C] location [O] próximo do louvre d...


# Hyperparams Search - Raw x MT5

In [ ]:
def tokenize_batch(batch):
  model_inputs = tokenizer(
        batch["input"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False
  )

  labels = tokenizer(
        batch["target"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False
  )

  model_inputs["labels"] = labels["input_ids"]

  return model_inputs

In [ ]:
MODEL_NAME = "google/mt5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

SPECIAL_TOKENS = ["[A]", "[C]", "[O]", "[P]", "[SSEP]"]
tokenizer.add_tokens(SPECIAL_TOKENS)

print("Tokenizer carregado.")
print("Tamanho do vocabulário:", len(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Tokenizer carregado.
Tamanho do vocabulário: 250105


In [ ]:
input_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["input"]
]

target_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["target"]
]

MAX_INPUT_LEN = int(np.percentile(input_lengths, 99))
MAX_TARGET_LEN = int(np.percentile(target_lengths, 99))

MAX_INPUT_LEN = max(MAX_INPUT_LEN, 64)
MAX_TARGET_LEN = max(MAX_TARGET_LEN, 64)

print("MAX_INPUT_LEN:", MAX_INPUT_LEN)
print("MAX_TARGET_LEN:", MAX_TARGET_LEN)

MAX_INPUT_LEN: 356
MAX_TARGET_LEN: 150


In [ ]:
def clean_pred(text):
  if not isinstance(text, str):
    return ""

  text = text.replace("<extra_id_0>", "")
  text = text.replace("<extra_id_1>", "")
  text = text.replace("<pad>", "")
  text = text.replace("</s>", "")
  text = " ".join(text.split())

  return text.strip()


def parse_target_string(text):
  if not isinstance(text, str):
    return set()

  text = clean_pred(text)

  quads = []

  for part in text.split("[SSEP]"):
      part = part.strip()

      if not part:
        continue

      try:
        a = part.split("[A]")[1].split("[C]")[0].strip()
        c = part.split("[C]")[1].split("[O]")[0].strip()
        o = part.split("[O]")[1].split("[P]")[0].strip()
        p = part.split("[P]")[1].strip()

        quads.append((a, c, o, p))

      except Exception:
        continue

  return set(quads)


def evaluate_asqp(golds, preds):
  total_correct = 0
  total_pred = 0
  total_gold = 0

  for gold, pred in zip(golds, preds):
    gold_set = parse_target_string(gold)
    pred_set = parse_target_string(pred)

    total_correct += len(gold_set & pred_set)
    total_pred += len(pred_set)
    total_gold += len(gold_set)

  precision = total_correct / total_pred if total_pred else 0
  recall = total_correct / total_gold if total_gold else 0

  f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
  )

  return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "correct": total_correct,
        "pred_total": total_pred,
        "gold_total": total_gold
    }

In [ ]:
def generate_predictions(model, tokenizer, texts):
  model.eval()
  preds = []

  for text in texts:
    inputs = tokenizer(
          text,
          return_tensors="pt",
          max_length=MAX_INPUT_LEN,
          truncation=True
    ).to(model.device)

    with torch.no_grad():
      outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LEN,
                num_beams=4,
                early_stopping=True)

      pred = tokenizer.decode(outputs[0], skip_special_tokens=False)
      pred = clean_pred(pred)

      preds.append(pred)

  return preds

In [ ]:
def objective(trial):
  clear_memory()

  learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        5e-4,
        log=True
  )

  num_train_epochs = 6

  weight_decay = trial.suggest_categorical(
        "weight_decay",
        [0.0, 0.001, 0.01]
  )

  warmup_ratio = trial.suggest_categorical(
        "warmup_ratio",
        [0.0, 0.05, 0.1]
  )

  kf = KFold(
        n_splits=3,
        shuffle=True,
        random_state=SEED
  )

  fold_f1s = []

  for fold, (train_idx, val_idx) in enumerate(kf.split(df), start=1):
      print(f"\nTrial {trial.number} | Fold {fold}/3")

      train_fold = df.iloc[train_idx].reset_index(drop=True)
      val_fold = df.iloc[val_idx].reset_index(drop=True)

      train_dataset = Dataset.from_pandas(train_fold)
      val_dataset = Dataset.from_pandas(val_fold)

      train_tokenized = train_dataset.map(
            tokenize_batch,
            batched=True,
            remove_columns=train_dataset.column_names
      )

      val_tokenized = val_dataset.map(
            tokenize_batch,
            batched=True,
            remove_columns=val_dataset.column_names
      )

      if MODEL_NAME == "unicamp-dl/ptt5-base-portuguese-vocab":

        model = T5ForConditionalGeneration.from_pretrained("unicamp-dl/ptt5-base-portuguese-vocab")
      else:

        model = AutoModelForSeq2SeqLM.from_pretrained(
              MODEL_NAME,
              tie_word_embeddings=False
        )

      model.resize_token_embeddings(len(tokenizer))
      model.to(DEVICE)

      data_collator = DataCollatorForSeq2Seq(
          tokenizer=tokenizer,
          model=model,
          label_pad_token_id=-100
      )

      fold_out_dir = f"/content/optuna_trial_{trial.number}_fold_{fold}"

      training_args = Seq2SeqTrainingArguments(
            output_dir=fold_out_dir,

            learning_rate=learning_rate,
            num_train_epochs=num_train_epochs,

            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            gradient_accumulation_steps= 1,

            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,

            predict_with_generate=True,
            generation_max_length=MAX_TARGET_LEN,

            eval_strategy="epoch",
            save_strategy="no",
            logging_strategy="epoch",

            fp16=False,
            report_to="none",

            seed=SEED,
            data_seed=SEED
      )

      trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            train_dataset=train_tokenized,
            eval_dataset=val_tokenized,
            data_collator=data_collator
      )

      trainer.train()

      preds = generate_predictions(
            model=model,
            tokenizer=tokenizer,
            texts=val_fold["input"].tolist()
      )

      golds = val_fold["target"].tolist()

      metrics = evaluate_asqp(golds, preds)
      fold_f1 = metrics["f1"]

      print("Fold metrics:", metrics)

      fold_f1s.append(fold_f1)

      del model
      del trainer
      del train_tokenized
      del val_tokenized
      clear_memory()

  mean_f1 = float(np.mean(fold_f1s))

  print(f"\nTrial {trial.number} | Mean CV F1:", mean_f1)

  return mean_f1

In [ ]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=6
)

print("Melhores hiperparâmetros:")
print(study.best_params)

print("Melhor F1 médio 3-fold:")
print(study.best_value)

[I 2026-06-24 03:14:37,843] A new study created in memory with name: no-name-d1c63b2d-117c-43c0-b2b0-959a0c755f73



Trial 0 | Fold 1/3


Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Map:   0%|          | 0/244 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,16.519315,3.273882
2,1.953048,0.710306
3,0.818822,0.571124
4,0.599714,0.497193
5,0.495424,0.491905
6,0.435181,0.487496


Fold metrics: {'precision': 0.5377155172413793, 'recall': 0.4324090121317158, 'f1': 0.47934678194044195, 'correct': 499, 'pred_total': 928, 'gold_total': 1154}


[I 2026-06-24 06:37:17,964] Trial 5 finished with value: 0.38007027902001517 and parameters: {'learning_rate': 0.00024395500114740264, 'weight_decay': 0.0, 'warmup_ratio': 0.05}. Best is trial 1 with value: 0.4918631435435004.



Trial 5 | Mean CV F1: 0.38007027902001517
Melhores hiperparâmetros:
{'learning_rate': 0.0001795816718972477, 'weight_decay': 0.0, 'warmup_ratio': 0.0}
Melhor F1 médio 3-fold:
0.4918631435435004


In [ ]:
optuna_results = study.trials_dataframe()

OPTUNA_PATH = "/content/drive/MyDrive/ASQP/saida_asqp/optuna_3fold_mt5_base_raw.csv"

optuna_results.to_csv(OPTUNA_PATH, index=False)

display(optuna_results)

print("Resultados salvos em:", OPTUNA_PATH)

,number,value,datetime_start,datetime_complete,duration,params_learning_rate,params_warmup_ratio,params_weight_decay,state
0,0,0.460821,2026-06-24 03:14:37.845123,2026-06-24 03:46:14.528586,0 days 00:31:36.683463,0.000186,0.05,0.010,COMPLETE
1,1,0.491863,2026-06-24 03:46:14.529511,2026-06-24 04:19:23.093266,0 days 00:33:08.563755,0.000180,0.00,0.000,COMPLETE
2,2,0.443130,2026-06-24 04:19:23.094244,2026-06-24 04:58:26.244734,0 days 00:39:03.150490,0.000118,0.05,0.000,COMPLETE
3,3,0.410647,2026-06-24 04:58:26.245768,2026-06-24 05:33:40.416009,0 days 00:35:14.170241,0.000114,0.05,0.010,COMPLETE
4,4,0.327527,2026-06-24 05:33:40.417157,2026-06-24 06:07:01.033349,0 days 00:33:20.616192,0.000203,0.10,0.001,COMPLETE
5,5,0.380070,2026-06-24 06:07:01.034532,2026-06-24 06:37:17.964752,0 days 00:30:16.930220,0.000244,0.05,0.000,COMPLETE


Resultados salvos em: /content/drive/MyDrive/ASQP/saida_asqp/optuna_3fold_mt5_base_raw.csv


# Hyperparams Search - Raw x PTT5

In [ ]:
MODEL_NAME = "unicamp-dl/ptt5-base-portuguese-vocab"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_tokens(SPECIAL_TOKENS)


print("Tokenizer carregado.")
print("Tamanho do vocabulário:", len(tokenizer))

Tokenizer carregado.
Tamanho do vocabulário: 32105


In [ ]:
input_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["input"]
]

target_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["target"]
]

MAX_INPUT_LEN = int(np.percentile(input_lengths, 99))
MAX_TARGET_LEN = int(np.percentile(target_lengths, 99))

MAX_INPUT_LEN = max(MAX_INPUT_LEN, 64)
MAX_TARGET_LEN = max(MAX_TARGET_LEN, 64)

print("MAX_INPUT_LEN:", MAX_INPUT_LEN)
print("MAX_TARGET_LEN:", MAX_TARGET_LEN)

MAX_INPUT_LEN: 274
MAX_TARGET_LEN: 169


In [ ]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=6
)

print("Melhores hiperparâmetros:")
print(study.best_params)

print("Melhor F1 médio 3-fold:")
print(study.best_value)

[I 2026-06-24 12:12:48,409] A new study created in memory with name: no-name-9e451224-d367-4d84-acea-b8eb5506595c



Trial 0 | Fold 1/3


Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Map:   0%|          | 0/244 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,7.556442,2.252323
2,2.145490,1.842410
3,1.750104,1.641838
4,1.508511,1.513558
5,1.347495,1.452454
6,1.279747,1.417243


Fold metrics: {'precision': 0.519774011299435, 'recall': 0.3986135181975737, 'f1': 0.4512015693967631, 'correct': 460, 'pred_total': 885, 'gold_total': 1154}


[I 2026-06-24 14:32:42,338] Trial 5 finished with value: 0.4203379845170842 and parameters: {'learning_rate': 0.0001709930427291189, 'weight_decay': 0.001, 'warmup_ratio': 0.05}. Best is trial 0 with value: 0.48807247721832003.



Trial 5 | Mean CV F1: 0.4203379845170842
Melhores hiperparâmetros:
{'learning_rate': 0.0004299009606724237, 'weight_decay': 0.01, 'warmup_ratio': 0.0}
Melhor F1 médio 3-fold:
0.48807247721832003


In [ ]:
optuna_results = study.trials_dataframe()

OPTUNA_PATH = "/content/drive/MyDrive/ASQP/saida_asqp/optuna_3fold_ptt5_base_raw.csv"

optuna_results.to_csv(OPTUNA_PATH, index=False)

display(optuna_results)

print("Resultados salvos em:", OPTUNA_PATH)

,number,value,datetime_start,datetime_complete,duration,params_learning_rate,params_warmup_ratio,params_weight_decay,state
0,0,0.488072,2026-06-24 12:12:48.410217,2026-06-24 12:38:34.406347,0 days 00:25:45.996130,0.000430,0.00,0.010,COMPLETE
1,1,0.297403,2026-06-24 12:38:34.407662,2026-06-24 12:59:57.706596,0 days 00:21:23.298934,0.000130,0.05,0.010,COMPLETE
2,2,0.478686,2026-06-24 12:59:57.707647,2026-06-24 13:25:37.488522,0 days 00:25:39.780875,0.000338,0.05,0.001,COMPLETE
3,3,0.446615,2026-06-24 13:25:37.489666,2026-06-24 13:47:59.178106,0 days 00:22:21.688440,0.000182,0.10,0.010,COMPLETE
4,4,0.467030,2026-06-24 13:47:59.179256,2026-06-24 14:10:33.878504,0 days 00:22:34.699248,0.000251,0.10,0.010,COMPLETE
5,5,0.420338,2026-06-24 14:10:33.879569,2026-06-24 14:32:42.338412,0 days 00:22:08.458843,0.000171,0.05,0.001,COMPLETE


Resultados salvos em: /content/drive/MyDrive/ASQP/saida_asqp/optuna_3fold_ptt5_base_raw.csv
